# A8: Address-Centric Database

**Purpose:** Create a master database with every Berkeley address as the foundation, linking all housing activity, news coverage, and future data sources.

## Architecture

```
                    ┌─────────────────────────┐
                    │   addresses (62,226)    │
                    │   Primary Key: apn      │
                    └───────────┬─────────────┘
                                │
         ┌──────────────────────┼──────────────────────┐
         │                      │                      │
         ▼                      ▼                      ▼
┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
│    projects     │   │  news_coverage  │   │  (future data)  │
│   FK: apn       │   │   FK: apn       │   │   FK: apn       │
└─────────────────┘   └─────────────────┘   └─────────────────┘
```

## Inputs
- `data/reference/berkeley_addresses_with_fields.csv` - 62,226 Berkeley addresses
- `outputs/housing_projects_comprehensive.csv` - 156 housing projects
- `outputs/gellerman_news_links.csv` - News coverage links

## Outputs
- `databases/berkeley_address_centric.db` - SQLite database
- `datasette-deploy/berkeley_address_centric.db` - For Datasette deployment

In [ ]:
# Environment Setup
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    if not os.path.exists('berkeley-housing-analysis'):
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
    ROOT = Path('berkeley-housing-analysis')
else:
    print('Running locally')
    ROOT = Path.cwd().parent if Path.cwd().name == '01_collection' else Path.cwd()

print(f'Project root: {ROOT}')

In [ ]:
import pandas as pd
import sqlite3
import re
from datetime import datetime

print('Imports successful')

## 1. Load Master Address Data

Load all 62,226 Berkeley addresses with APNs, coordinates, and street information.

In [ ]:
# Load Berkeley addresses
addresses_path = ROOT / 'data/reference/berkeley_addresses_with_fields.csv'
df_addresses = pd.read_csv(addresses_path)

print(f'Loaded {len(df_addresses):,} Berkeley addresses')
print(f'\nColumns: {list(df_addresses.columns)}')
df_addresses.head()

In [ ]:
# Clean and standardize address data
def normalize_address(addr):
    """Normalize address for matching"""
    if pd.isna(addr):
        return None
    addr = str(addr).upper().strip()
    # Remove unit numbers for base address matching
    addr = re.sub(r'\s*(APT|UNIT|STE|#)\s*\S+$', '', addr)
    # Standardize street types
    replacements = {
        ' STREET': ' ST', ' AVENUE': ' AVE', ' BOULEVARD': ' BLVD',
        ' DRIVE': ' DR', ' ROAD': ' RD', ' LANE': ' LN',
        ' COURT': ' CT', ' PLACE': ' PL', ' WAY': ' WAY'
    }
    for old, new in replacements.items():
        addr = addr.replace(old, new)
    return addr

# Create normalized address column
df_addresses['address_normalized'] = df_addresses['ADDRESS'].apply(normalize_address)

# Create base address (without unit) for grouping
def extract_base_address(row):
    """Create base address from components"""
    parts = []
    if pd.notna(row['ST_NUM']):
        parts.append(str(int(row['ST_NUM'])))
    if pd.notna(row['FEANME']):
        parts.append(str(row['FEANME']).upper())
    if pd.notna(row['FEATYP']):
        parts.append(str(row['FEATYP']).upper())
    return ' '.join(parts) if parts else None

df_addresses['base_address'] = df_addresses.apply(extract_base_address, axis=1)

print(f'Sample normalized addresses:')
print(df_addresses[['ADDRESS', 'address_normalized', 'base_address']].head(10))

In [ ]:
# Create clean addresses table for database
# Select and rename columns for clarity

addresses_clean = df_addresses[[
    'APN', 'ADDRESS', 'address_normalized', 'base_address',
    'ST_NUM', 'FEANME', 'FEATYP', 'UNIT', 'UNIT_TYP',
    'CITY', 'ZIPCODE', 'latitude', 'longitude'
]].copy()

addresses_clean.columns = [
    'apn', 'address_full', 'address_normalized', 'base_address',
    'street_number', 'street_name', 'street_type', 'unit', 'unit_type',
    'city', 'zipcode', 'latitude', 'longitude'
]

# Add unique ID
addresses_clean.insert(0, 'address_id', range(1, len(addresses_clean) + 1))

print(f'\nPrepared {len(addresses_clean):,} addresses for database')
print(f'\nUnique APNs: {addresses_clean["apn"].nunique():,}')
print(f'Unique base addresses: {addresses_clean["base_address"].nunique():,}')
print(f'Unique zipcodes: {sorted(addresses_clean["zipcode"].dropna().unique())}')

addresses_clean.head()

## 2. Load Housing Projects

In [ ]:
# Load comprehensive housing projects
projects_path = ROOT / 'outputs/housing_projects_comprehensive.csv'

if projects_path.exists():
    df_projects = pd.read_csv(projects_path)
    print(f'Loaded {len(df_projects)} projects from comprehensive dataset')
else:
    # Fallback to FINAL
    projects_path = ROOT / 'data/processed/housing_projects_FINAL.csv'
    df_projects = pd.read_csv(projects_path)
    print(f'Loaded {len(df_projects)} projects from FINAL dataset')

df_projects.head()

In [ ]:
# Match projects to master addresses
# Strategy: Match on normalized address, then by APN if available

def find_matching_address(project_addr, addresses_df):
    """Find matching address_id from master table"""
    if pd.isna(project_addr):
        return None
    
    # Normalize project address
    normalized = normalize_address(project_addr)
    
    # Try exact match on base_address
    matches = addresses_df[addresses_df['base_address'] == normalized]
    if len(matches) > 0:
        return matches.iloc[0]['address_id']
    
    # Try matching on address_normalized
    matches = addresses_df[addresses_df['address_normalized'].str.contains(normalized, na=False, regex=False)]
    if len(matches) > 0:
        return matches.iloc[0]['address_id']
    
    return None

# Match projects to addresses
print('Matching projects to master addresses...')
address_col = 'address_normalized' if 'address_normalized' in df_projects.columns else 'address_display'

matched_ids = []
for idx, row in df_projects.iterrows():
    addr = row.get(address_col) or row.get('address_display')
    match_id = find_matching_address(addr, addresses_clean)
    matched_ids.append(match_id)

df_projects['address_id'] = matched_ids

matched = df_projects['address_id'].notna().sum()
print(f'\nMatched {matched}/{len(df_projects)} projects to master addresses ({matched/len(df_projects)*100:.1f}%)')

In [ ]:
# Prepare projects table for database
# Select relevant columns

project_columns = ['project_id', 'address_id', 'address_display']

# Add optional columns if they exist
optional_cols = [
    'apn', 'net_units', 'new_units', 'old_units', 'year',
    'status', 'permits', 'description', 'data_source',
    'has_official_permit', 'has_media_coverage',
    'latitude', 'longitude', 'primary_news_source'
]

for col in optional_cols:
    if col in df_projects.columns:
        project_columns.append(col)

# Rename project_id if needed
if 'project_id' not in df_projects.columns:
    if 'id' in df_projects.columns:
        df_projects['project_id'] = df_projects['id']
    else:
        df_projects['project_id'] = range(1, len(df_projects) + 1)

projects_clean = df_projects[[c for c in project_columns if c in df_projects.columns]].copy()

print(f'Prepared {len(projects_clean)} projects with columns:')
print(list(projects_clean.columns))

## 3. Load News Coverage

In [ ]:
# Load news links
news_path = ROOT / 'outputs/gellerman_news_links.csv'

if news_path.exists():
    df_news = pd.read_csv(news_path)
    print(f'Loaded {len(df_news)} news links')
    
    # Match news to projects (which link to addresses)
    # News links have project_name which maps to project address
    print(f'\nNews sources: {df_news["source_category"].value_counts().to_dict()}')
    df_news.head()
else:
    print('No news links file found')
    df_news = pd.DataFrame(columns=['project_name', 'url', 'source_category', 'address_normalized'])

In [ ]:
# Create news_coverage table linked to projects
# Match news items to projects by address

if len(df_news) > 0:
    news_records = []
    news_id = 1
    
    for idx, row in df_news.iterrows():
        # Try to find matching project
        project_match = None
        
        if pd.notna(row.get('address_normalized')):
            addr = row['address_normalized']
            matches = projects_clean[
                projects_clean['address_display'].str.upper().str.contains(
                    addr.split()[0] if addr else '', na=False
                )
            ]
            if len(matches) > 0:
                project_match = matches.iloc[0]['project_id']
        
        news_records.append({
            'news_id': news_id,
            'project_id': project_match,
            'project_name': row.get('project_name'),
            'url': row.get('url'),
            'source': row.get('source_category'),
            'date_added': datetime.now().strftime('%Y-%m-%d')
        })
        news_id += 1
    
    news_clean = pd.DataFrame(news_records)
    linked = news_clean['project_id'].notna().sum()
    print(f'Created {len(news_clean)} news records, {linked} linked to projects')
else:
    news_clean = pd.DataFrame(columns=['news_id', 'project_id', 'project_name', 'url', 'source', 'date_added'])
    print('No news data to process')

## 4. Create SQLite Database

In [ ]:
# Create database
db_dir = ROOT / 'databases'
db_dir.mkdir(exist_ok=True)

db_path = db_dir / 'berkeley_address_centric.db'

# Remove existing database
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f'Creating database: {db_path}')

In [ ]:
# Create addresses table (master table)
addresses_clean.to_sql('addresses', conn, if_exists='replace', index=False)

# Create indexes for fast lookups
cursor.execute('CREATE INDEX idx_addresses_apn ON addresses(apn)')
cursor.execute('CREATE INDEX idx_addresses_base ON addresses(base_address)')
cursor.execute('CREATE INDEX idx_addresses_normalized ON addresses(address_normalized)')
cursor.execute('CREATE INDEX idx_addresses_coords ON addresses(latitude, longitude)')
cursor.execute('CREATE INDEX idx_addresses_zipcode ON addresses(zipcode)')
cursor.execute('CREATE INDEX idx_addresses_street ON addresses(street_name)')

print(f'Created addresses table with {len(addresses_clean):,} rows')

In [ ]:
# Create projects table
projects_clean.to_sql('projects', conn, if_exists='replace', index=False)

# Create indexes
cursor.execute('CREATE INDEX idx_projects_address_id ON projects(address_id)')
if 'apn' in projects_clean.columns:
    cursor.execute('CREATE INDEX idx_projects_apn ON projects(apn)')
if 'status' in projects_clean.columns:
    cursor.execute('CREATE INDEX idx_projects_status ON projects(status)')
if 'year' in projects_clean.columns:
    cursor.execute('CREATE INDEX idx_projects_year ON projects(year)')

print(f'Created projects table with {len(projects_clean)} rows')

In [ ]:
# Create news_coverage table
news_clean.to_sql('news_coverage', conn, if_exists='replace', index=False)

# Create indexes
cursor.execute('CREATE INDEX idx_news_project_id ON news_coverage(project_id)')
cursor.execute('CREATE INDEX idx_news_source ON news_coverage(source)')

print(f'Created news_coverage table with {len(news_clean)} rows')

In [ ]:
# Create summary statistics table
stats = {
    'total_addresses': len(addresses_clean),
    'unique_apns': addresses_clean['apn'].nunique(),
    'total_projects': len(projects_clean),
    'total_units': int(projects_clean['net_units'].sum()) if 'net_units' in projects_clean.columns else 0,
    'total_news_links': len(news_clean),
    'generated': datetime.now().isoformat()
}

pd.DataFrame([stats]).to_sql('database_stats', conn, if_exists='replace', index=False)
print(f'Created database_stats table')

conn.commit()
print('\nDatabase committed successfully')

## 5. Create Useful Views

In [ ]:
# Create views for common queries

# View: Addresses with housing projects
cursor.execute('''
CREATE VIEW IF NOT EXISTS addresses_with_projects AS
SELECT 
    a.*,
    p.project_id,
    p.net_units,
    p.status as project_status,
    p.year as project_year,
    p.data_source
FROM addresses a
LEFT JOIN projects p ON a.address_id = p.address_id
''')

# View: Projects with full address details
cursor.execute('''
CREATE VIEW IF NOT EXISTS projects_full AS
SELECT 
    p.*,
    a.address_full,
    a.zipcode,
    a.street_name,
    a.street_type
FROM projects p
LEFT JOIN addresses a ON p.address_id = a.address_id
''')

# View: Street-level summary
cursor.execute('''
CREATE VIEW IF NOT EXISTS streets_summary AS
SELECT 
    street_name,
    street_type,
    COUNT(DISTINCT address_id) as address_count,
    COUNT(DISTINCT apn) as parcel_count,
    MIN(latitude) as min_lat,
    MAX(latitude) as max_lat
FROM addresses
WHERE street_name IS NOT NULL
GROUP BY street_name, street_type
ORDER BY address_count DESC
''')

# View: Development activity by street
cursor.execute('''
CREATE VIEW IF NOT EXISTS development_by_street AS
SELECT 
    a.street_name,
    a.street_type,
    COUNT(DISTINCT p.project_id) as project_count,
    SUM(p.net_units) as total_units,
    GROUP_CONCAT(DISTINCT p.status) as statuses
FROM addresses a
INNER JOIN projects p ON a.address_id = p.address_id
WHERE a.street_name IS NOT NULL
GROUP BY a.street_name, a.street_type
ORDER BY total_units DESC
''')

conn.commit()
print('Created database views:')
print('  - addresses_with_projects')
print('  - projects_full')
print('  - streets_summary')
print('  - development_by_street')

## 6. Test Queries

In [ ]:
# Test: Look up a specific address
test_address = '2276 SHATTUCK'

query = f'''
SELECT 
    a.address_full,
    a.apn,
    a.zipcode,
    p.net_units,
    p.status,
    p.data_source
FROM addresses a
LEFT JOIN projects p ON a.address_id = p.address_id
WHERE a.base_address LIKE '%{test_address}%'
LIMIT 10
'''

result = pd.read_sql(query, conn)
print(f'Query: Find addresses matching "{test_address}"')
print(result)

In [ ]:
# Test: Get all addresses on a street with development activity
street = 'SHATTUCK'

query = f'''
SELECT * FROM development_by_street
WHERE street_name = '{street}'
'''

result = pd.read_sql(query, conn)
print(f'Development activity on {street}:')
print(result)

In [ ]:
# Test: Top streets by number of addresses
query = '''
SELECT * FROM streets_summary
LIMIT 20
'''

result = pd.read_sql(query, conn)
print('Top 20 streets by address count:')
print(result)

In [ ]:
# Test: Database statistics
print('DATABASE STATISTICS')
print('='*50)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f'\nTables:')
for table in tables['name']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {table}', conn).iloc[0]['n']
    print(f'  {table}: {count:,} rows')

views = pd.read_sql("SELECT name FROM sqlite_master WHERE type='view'", conn)
print(f'\nViews:')
for view in views['name']:
    print(f'  {view}')

# File size
import os
size_mb = os.path.getsize(db_path) / (1024 * 1024)
print(f'\nDatabase size: {size_mb:.1f} MB')

## 7. Copy to Datasette Deploy

In [ ]:
# Close connection before copying
conn.close()

# Copy to datasette-deploy directory
import shutil

datasette_dir = ROOT / 'datasette-deploy'
datasette_db = datasette_dir / 'berkeley_address_centric.db'

shutil.copy(db_path, datasette_db)
print(f'Copied database to: {datasette_db}')
print(f'Size: {os.path.getsize(datasette_db) / (1024*1024):.1f} MB')

In [ ]:
# Create metadata for address-centric database
import json

metadata = {
    "title": "Berkeley Address-Centric Database",
    "description": f"Every Berkeley address ({len(addresses_clean):,}) linked to housing projects, news coverage, and development activity.",
    "source": "Alameda County GIS, City of Berkeley Planning, Community Sources",
    "source_url": "https://github.com/blockXblock/berkeley-housing-analysis",
    "databases": {
        "berkeley_address_centric": {
            "title": "Berkeley Addresses",
            "description": f"{len(addresses_clean):,} addresses, {len(projects_clean)} housing projects",
            "tables": {
                "addresses": {
                    "title": f"All Addresses ({len(addresses_clean):,})",
                    "description": "Every Berkeley address with APN and coordinates",
                    "sort_desc": "address_id",
                    "facets": ["zipcode", "street_name"],
                    "plugins": {
                        "datasette-cluster-map": {
                            "latitude_column": "latitude",
                            "longitude_column": "longitude"
                        }
                    }
                },
                "projects": {
                    "title": f"Housing Projects ({len(projects_clean)})",
                    "description": "Active housing development projects",
                    "sort_desc": "net_units",
                    "facets": ["status", "data_source", "year"]
                },
                "news_coverage": {
                    "title": "News Coverage",
                    "description": "Media articles about housing projects",
                    "facets": ["source"]
                }
            },
            "queries": {
                "address_lookup": {
                    "title": "Address Lookup",
                    "description": "Find an address and see all associated data",
                    "sql": "SELECT a.*, p.net_units, p.status, p.year FROM addresses a LEFT JOIN projects p ON a.address_id = p.address_id WHERE a.base_address LIKE '%' || :address || '%' LIMIT 50"
                },
                "street_development": {
                    "title": "Street Development Activity",
                    "description": "Housing development by street",
                    "sql": "SELECT * FROM development_by_street ORDER BY total_units DESC"
                },
                "addresses_with_projects": {
                    "title": "Addresses with Housing Projects",
                    "description": "All addresses that have housing development",
                    "sql": "SELECT a.address_full, a.apn, a.zipcode, p.net_units, p.status, p.year FROM addresses a INNER JOIN projects p ON a.address_id = p.address_id ORDER BY p.net_units DESC"
                },
                "zipcode_summary": {
                    "title": "Summary by Zipcode",
                    "description": "Address and project counts by zipcode",
                    "sql": "SELECT zipcode, COUNT(*) as addresses, COUNT(DISTINCT street_name) as streets FROM addresses GROUP BY zipcode ORDER BY addresses DESC"
                }
            }
        }
    },
    "plugins": {
        "datasette-cluster-map": {
            "latitude_column": "latitude",
            "longitude_column": "longitude"
        }
    }
}

metadata_path = datasette_dir / 'metadata_address_centric.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Created metadata: {metadata_path}')

## 8. Summary

The address-centric database is now ready!

In [ ]:
print('ADDRESS-CENTRIC DATABASE COMPLETE')
print('='*60)
print(f'\nDatabase: {db_path}')
print(f'Size: {os.path.getsize(db_path) / (1024*1024):.1f} MB')
print(f'\nTables:')
print(f'  - addresses: {len(addresses_clean):,} Berkeley addresses')
print(f'  - projects: {len(projects_clean)} housing projects')
print(f'  - news_coverage: {len(news_clean)} news links')
print(f'\nViews:')
print(f'  - addresses_with_projects')
print(f'  - projects_full')
print(f'  - streets_summary')
print(f'  - development_by_street')
print(f'\nTo deploy to Datasette:')
print(f'  cd datasette-deploy')
print(f'  fly deploy')
print('='*60)